In [7]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [8]:
data = {
    "Gr_Studies": ["No", "Yes", "No", "No", "No", "Yes", "No", "Yes", "No", "Yes", "No", "No"],
    "Family_Wealth": ["Poor", "Rich", "Rich", "Poor", "Rich", "Middle", "Rich", "Middle", "Poor", "Poor", "Poor", "Poor"],
    "Risk": ["RA", "RS", "RA", "RS", "RA", "RS", "RA", "RS", "RS", "RS", "RS", "RS"],
    "Duration": ["M", "Q", "F", "M", "M", "F", "M", "F", "F", "M", "M", "Q"]
}

import pandas as pd
df = pd.DataFrame(data)
df.columns = ['Gr_Studies', 	'Family_Wealth',	'Risk',	'Duration']

df

,Gr_Studies,Family_Wealth,Risk,Duration
0,No,Poor,RA,M
1,Yes,Rich,RS,Q
2,No,Rich,RA,F
3,No,Poor,RS,M
4,No,Rich,RA,M
5,Yes,Middle,RS,F
6,No,Rich,RA,M
7,Yes,Middle,RS,F
8,No,Poor,RS,F
9,Yes,Poor,RS,M


In [9]:
df = df.rename(columns={
    "Gr_Studies":"grad_studies",
    "Family_Wealth":"family_wealth",
    "Duration" : "duration",
    "Risk" : "risk"
})

df["duration"].value_counts()

,count
duration,
M,6
F,4
Q,2


In [14]:
#compare information gain
def entropy(y):
    counts = y.value_counts(normalize=True)
    return -np.sum(counts*np.log2(counts))

def weighted_entropy(groups, target_col):
    total = sum(len(g) for g in groups)
    return sum((len(g)/total)*entropy(g[target_col]) for g in groups)

def information_gain(df, groups, target_col):
    return entropy(df[target_col]) - weighted_entropy(groups, target_col)

In [13]:
groups_gr = [df[df['grad_studies'] == val] for val in df['grad_studies'].unique()]
groups_fw = [df[df['family_wealth'] == val] for val in df['family_wealth'].unique()]
groups_risk = [df[df['risk'] == val] for val in df['risk'].unique()]

In [15]:
ig_gr = information_gain(df, groups_gr, 'duration')
ig_fw = information_gain(df, groups_fw, 'duration')
ig_risk = information_gain(df, groups_risk, 'duration')

In [17]:
print(f"IG(Gr_Studies) = {ig_gr:.4f}")
print(f"IG(Family_Wealth) = {ig_fw:.4f}")
print(f"IG(Risk) = {ig_risk:.4f}")

best = max([('Gr_Studies', ig_gr), ('Family_Wealth', ig_fw), ('Risk', ig_risk)], key=lambda x: x[1])
print(f"\nBest attribute = {best[0]} with IG = {best[1]:.4f}")

IG(Gr_Studies) = 0.0933
IG(Family_Wealth) = 0.3333
IG(Risk) = 0.1479

Best attribute = Family_Wealth with IG = 0.3333


In [18]:
for val in df['family_wealth'].unique():
    group = df[df['family_wealth'] == val]
    prediction = group['duration'].value_counts().idxmax()
    print(f"Family_Wealth={val}: prediction={prediction}, counts={group['duration'].value_counts().to_dict()}")

Family_Wealth=Poor: prediction=M, counts={'M': 4, 'F': 1, 'Q': 1}
Family_Wealth=Rich: prediction=M, counts={'M': 2, 'Q': 1, 'F': 1}
Family_Wealth=Middle: prediction=F, counts={'F': 2}


In [19]:
def predict(row):
    if row['family_wealth'] == 'Poor':
        return 'M'
    elif row['family_wealth'] == 'Middle':
        return 'F'
    else:  # Rich
        return 'M'

df['predicted'] = df.apply(predict, axis=1)
print(df[['duration', 'predicted']])

   duration predicted
0         M         M
1         Q         M
2         F         M
3         M         M
4         M         M
5         F         F
6         M         M
7         F         F
8         F         M
9         M         M
10        M         M
11        Q         M


In [20]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

y_true = df['duration']
y_pred = df['predicted']

print(f"Accuracy = {accuracy_score(y_true, y_pred):.4f}")
print(f"Recall (macro) = {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"Recall (micro) = {recall_score(y_true, y_pred, average='micro'):.4f}")
print(f"Precision (macro) = {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Precision (micro) = {precision_score(y_true, y_pred, average='micro'):.4f}")
print(f"F1 (macro) = {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1 (micro) = {f1_score(y_true, y_pred, average='micro'):.4f}")

Accuracy = 0.6667
Recall (macro) = 0.5000
Recall (micro) = 0.6667
Precision (macro) = 0.5333
Precision (micro) = 0.6667
F1 (macro) = 0.4722
F1 (micro) = 0.6667


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
print("Decision Tree (depth=1)")
print("Root: Family_Wealth (IG=0.3333)")
print("|")
for val in ['Poor', 'Middle', 'Rich']:
    group = df[df['family_wealth'] == val]
    prediction = group['duration'].value_counts().idxmax()
    print(f"|-- Family_Wealth = {val} --> Predict: {prediction} ")

Decision Tree (depth=1)
Root: Family_Wealth (IG=0.3333)
|
|-- Family_Wealth = Poor --> Predict: M 
|-- Family_Wealth = Middle --> Predict: F 
|-- Family_Wealth = Rich --> Predict: M 
